In [4]:
# Cell 0: Dependency Installation
# Uncomment and run these lines once to install the required packages:
#   - langchain-openai : LangChain wrapper for OpenAI models (GPT-4.1, o3-mini, etc.)
#   - openpyxl         : Read/write Excel files (.xlsx)
#   - boto3            : AWS SDK used to call models via Amazon Bedrock
#   - langchain-ollama : LangChain wrapper for locally-hosted Ollama models
#!pip install langchain-openai
#!pip install openpyxl
#!pip install boto3
#!pip install langchain-ollama


In [5]:
# Cell 1: Library Imports
# LangChain model wrappers for OpenAI (GPT) and Ollama (local) inference
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
import re                          # Regex for parsing <think>…</think> tags
import pandas as pd                # DataFrame operations and Excel I/O
import getpass                     # Secure prompt for manual key entry if needed
import os
from dotenv import load_dotenv     # Load secrets from a local .env file
from tqdm import tqdm              # Progress bar for LLM calls
import tiktoken                    # Token counting for OpenAI-compatible models
import time                        # Measure wall-clock execution time
import json
import boto3                       # AWS SDK for Amazon Bedrock model access
from botocore.exceptions import ClientError


In [6]:
# Cell 2: API Keys and AWS Session Initialization
#
# OpenAI key is read from a .env file (OPENAI_API_KEY=...) and injected into
# the environment so LangChain picks it up automatically.
#
# AWS credentials for Amazon Bedrock must also be stored in the .env file or
# configured via the AWS CLI / IAM roles.  NEVER hard-code credentials here.
# Example .env entries:
#   OPENAI_API_KEY=sk-...
#   AWS_ACCESS_KEY_ID=AKIA...
#   AWS_SECRET_ACCESS_KEY=...
#   AWS_DEFAULT_REGION=us-west-2

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Create a reusable boto3 session for Amazon Bedrock (region: us-west-2).
# Credentials are read from environment variables set by load_dotenv() above.
session = boto3.Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_DEFAULT_REGION", "us-west-2")
)


In [7]:
# Cell 3: Load the Paper Dataset
# Reads an Excel file containing 7 academic papers on scientific retractions.
# Each row represents one paper with the following columns:
#   Paper_Id           : Unique identifier (e.g. 'paper1')
#   Title              : Paper title
#   Abstract           : Paper abstract
#   Authors            : Semicolon-separated author names
#   Affiliations       : Author institutional affiliations
#   Year               : Publication year
#   Related_Work       : Raw 'Related Work' section text
#   Related_Work_Final : Cleaned version of Related_Work used as LLM input
#   Citation_Standard  : Numeric citation index (e.g. '[338]')
df = pd.read_excel('data/Test_Jodi_Overallv01_Inter.xlsx')  # Update path as needed
df


,Paper_Id,Title,Abstract,Authors,Affiliations,Year,Related_Work,Related_Work_Final,Citation_Standard
0,paper7,why do some retracted articles continue to get...,retracted publications can still receive a sub...,marion schmidt,german center for higher education research an...,2024,retractions of scientific publications are oft...,retractions of scientific publications are oft...,[338]
1,paper3,what difference might retractions make? an est...,the extent to which a retraction might require...,daniele fanelli; julie wong; david moher,"centre for journalology, clinical epidemiology...",2021,retractions are a phenomenon of growing import...,retractions are a phenomenon of growing import...,[332]
2,paper2,addressing unreliability propagation in scient...,today's scientists rely on scientific artifact...,heng zheng; yuanxi fu; m sarol; ishita sarraf;...,"grinnell college grinnell, usa; jodi schneider...",2024,to identify publications that propagate unreli...,to identify publications that propagate unreli...,[339]
3,paper6,critical citations in knowledge construction a...,critical citations are lacking a common defini...,frederique bordignon,"ecole des ponts, marne-la-vallée, france",2022,"in studies about knowledge construction, criti...","in studies about knowledge construction, criti...",[340]
4,paper4,continued use of retracted papers: temporal tr...,we present the first database-wide study on th...,tzu-kun hsiao; jodi schneider,"school of information sciences, university of ...",2021,citation context analysis has been used to exp...,citation context analysis has been used to exp...,[114]
5,paper1,towards knowledge maintenance in scientific di...,scientific digital libraries speed disseminati...,yuanxi fu; jodi schneider,"school of information sciences, university of ...",2020,argumentation theory is an interdisciplinary f...,argumentation theory is an interdisciplinary f...,[45]
6,paper5,continued post-retraction citation of a fraudu...,this paper presents a case study of long-term ...,jodi schneider; di ye; alison hill; ashley whi...,"data science institute, columbia university, n...",2020,retraction is intended to remove seriously fla...,retraction is intended to remove seriously fla...,[8]


In [8]:
# Cell 4: Helper Function – extract_think_text()
# Some reasoning models (e.g. DeepSeek-R1, Ollama-based models) wrap their
# internal chain-of-thought inside <think>…</think> tags before the final answer.
# This function splits the raw response into:
#   between_think : The model's internal reasoning (may be empty for non-thinking models)
#   after_think   : The actual coreference-resolved text

def extract_think_text(text):
    match = re.search(r"<think>(.*?)</think>(.*)", text, re.DOTALL)
    if match:
        between_think = match.group(1).strip()  # Internal reasoning block
        after_think = match.group(2).strip()    # Final resolved text
    else:
        # Model returned plain text without a thinking block
        between_think = ""
        after_think = ""
    
    return between_think, after_think


In [9]:
# Cell 5: Coreference Resolution Prompt Templates
#
# 'prompt'         – Main prompt template used for all models (GPT, Claude, DeepSeek, LLaMA).
#                    The placeholder '_context_' is replaced at runtime with the actual
#                    Related_Work text for each paper.
#
# 'prompt_mistral' – Mistral-specific variant that wraps the prompt in the
#                    <s>[INST]…[/INST] instruction format required by Mistral models.
#
# The prompt instructs the model to:
#   1. Replace all pronouns (personal, possessive, reflexive, demonstrative, relative)
#      with their explicit referents.
#   2. Expand ambiguous definite noun phrases (e.g. 'the experiment') with specific names.
#   3. Preserve all punctuation, spacing, citations ([#]), and paragraph markers
#      ('.[line break].') exactly as in the input.
#   4. Self-check formatting before producing the final output.
#
# The first (commented-out) block is an earlier version of the prompt kept for reference.

# prompt= f"""
# <context>
# Please perform coreference resolution on the scientific text provided below. Analyze the provided scientific text and replace ambiguous references with their explicit antecedents to create a version where all coreferences are fully resolved. The goal is to produce a modified text where each mention clearly and explicitly refers to its intended entity, making the text more precise and easier to understand.
# </context>

# <scientific text>
# _context_
# </scientific text>

# <instructions>
# - Replace personal pronouns (I, you, he, she, it, we, they, me, you, him, her, it, us, them) with their full referents.
# - Replace possessive pronouns (my, your, his, her, its, our, their, mine, yours, his, hers, its, ours, theirs) with their full referents.
# - Replace reflexive pronouns (myself, yourself, himself, herself, itself, ourselves, yourselves, themselves) with their full referents.
# - Replace demonstrative pronouns (this, that, these, those) with their full referents.
# - Replace relative pronouns (who, whom, whose, which, that) with their full referents.
# - Substitute definite noun phrases (e.g., "the experiment", "the model") with more specific identifiers.
# - Expand abbreviations where first introduced, then maintain consistency.
# - Replace demonstrative references ("this effect," "these results") with explicit entities.
# - Maintain scientific accuracy while improving clarity.
# </instructions>

# <special considerations>
# - References to other works are represented as "[#]". For instance, "[12] applied yoga breathing techniques." "[#]" could refer to a single work or several works.
# - The text ".[line break]." indicates that the next text belongs to a new paragraph.
# - Preserve technical precision and domain-specific terminology.
# - Maintain the original meaning and scientific integrity of the content.
# - Ensure consistent reference resolution across different paragraphs.
# - Keep mathematical notations and equations intact.
# - Do not paraphrase, rephrase, or change any part of the original scientific text except for resolving coreferences.
# - Do not change the punctuation, formatting of the original scientific text.
# - Don not change the text ".[line break].".
# </special considerations>

# <output guidelines>
# - Your output should be the modified text with all the coreferences resolved.
# - Only show the resulting text; do not include any explanation and any tag.
# - Do not omit any text from the original scientific text.
# - Keep the text ".[line break].".
# </output guidelines>

# """


prompt=f"""
<context>
Please perform coreference resolution on the scientific text provided below. Analyze the provided scientific text and replace ambiguous references with their explicit antecedents to create a version where all coreferences are fully resolved. The goal is to produce a modified text where each mention clearly and explicitly refers to its intended entity, making the text more precise and easier to understand.
</context>

<scientific text>
_context_
</scientific text>

<instructions>
- Replace personal pronouns (I, you, he, she, it, we, they, me, you, him, her, it, us, them) with their full referents.
- Replace possessive pronouns (my, your, his, her, its, our, their, mine, yours, his, hers, its, ours, theirs) with their full referents.
- Replace reflexive pronouns (myself, yourself, himself, herself, itself, ourselves, yourselves, themselves) with their full referents.
- Replace demonstrative pronouns (this, that, these, those) with their full referents.
- Replace relative pronouns (who, whom, whose, which, that) with their full referents.
- Substitute definite noun phrases (e.g., "the experiment", "the model") with more specific identifiers.
- Expand abbreviations where first introduced, then maintain consistency.
- Replace demonstrative references ("this effect," "these results") with explicit entities.
- Maintain scientific accuracy while improving clarity.

**Critical formatting rules:**
- Do not add, remove, or alter *any punctuation marks* (including commas, periods, colons, semicolons, hyphens, quotation marks, parentheses, brackets, etc.).
- Do not change spacing, capitalization, or line breaks.
- Do not normalize punctuation (e.g., do not replace three dots with an ellipsis symbol).
- Preserve the text exactly as given, except for explicit coreference resolution.

**Self-check requirement before final output:**
- Compare the punctuation, line breaks, and spacing of the input text with the modified text.
- If any punctuation, spacing, or line breaks differ (other than explicit coreference replacements), correct them before finalizing.
- Only release the final output after confirming that punctuation and formatting are identical to the input.
</instructions>

<special considerations>
- References to other works are represented as "[#]". For instance, "[12] applied yoga breathing techniques." "[#]" could refer to a single work or several works.
- The text ".[line break]." indicates that the next text belongs to a new paragraph.
- Preserve technical precision and domain-specific terminology.
- Maintain the original meaning and scientific integrity of the content.
- Ensure consistent reference resolution across different paragraphs.
- Keep mathematical notations and equations intact.
- Do not paraphrase, rephrase, or change any part of the original scientific text except for resolving coreferences.
- Do not change the punctuation, formatting, spacing, or line breaks of the original scientific text.
- Do not change the text ".[line break].".
</special considerations>

<output guidelines>
- Your output should be the modified text with all the coreferences resolved.
- Only show the resulting text; do not include any explanation and any tag.
- Do not omit any text from the original scientific text.
- Keep the text ".[line break].".
- Preserve punctuation, formatting, and line breaks exactly as in the input.
- Ensure that the punctuation, spacing, and line breaks of the output match the input (other than necessary coreference replacements).
</output guidelines>

"""


# Prompt with special tags
prompt_mistral="<s>[INST] "+prompt+" [/INST]"    

In [10]:
# Cell 6: Coreference Resolution – OpenAI / Ollama Models
#
# get_coreference_resolution_others(context, model_name, model_mixed)
#
# Handles inference for models served via:
#   - OpenAI API  : any model name containing 'gpt' (e.g. 'gpt-4.1-2025-04-14')
#   - OpenAI API  : 'o3-mini' (reasoning model; temperature is left at default=1)
#   - Ollama      : all other model names (locally hosted models)
#
# Parameters:
#   context      : The Related_Work_Final text for a single paper
#   model_name   : LLM identifier string
#   model_mixed  : 'YES' to enable extended thinking (Claude only), 'NO' otherwise
#
# Returns:
#   thinking      : Internal reasoning text (empty string for GPT / o3-mini)
#   response_final: Coreference-resolved text
#   num_tokens    : Number of tokens in the prompt (cl100k_base encoding)
#   elapsed_time  : Wall-clock seconds for the LLM call
#
# Token counting uses tiktoken:
#   - gpt-4o models : model-specific encoding
#   - all others    : cl100k_base (compatible with GPT-4.1 and most modern models)

def get_coreference_resolution_others(context: str, model_name:str,model_mixed:str):
    
    model_mixed= model_mixed.upper()
 
    if model_mixed=="YES" or model_mixed=="NO": 
    
        start_time = time.time()  # Start timing
        
        # Initialize the chat model based on the specified model name
        if "gpt" in model_name:
            llm_generator = ChatOpenAI(model_name=model_name, temperature=0) # Temperature=0.
            if "gpt-4o" in model_name:
                encoding = tiktoken.encoding_for_model(model_name)
            else:
                encoding = tiktoken.get_encoding("cl100k_base") # For gpt-4.1
                
        else:
            if "o3-mini" in model_name:
                llm_generator = ChatOpenAI(model_name=model_name, temperature=None) # Default Temperature=1.
                encoding = tiktoken.get_encoding("cl100k_base")
            else:
                llm_generator = ChatOllama(model=model_name, temperature=0) # Temperature=0.
                encoding = tiktoken.get_encoding("cl100k_base")
        
        # Prompt
        user_question =  prompt.replace("_context_", context) 
     
       
        # Calculate number of tokens
        num_tokens = len(encoding.encode(user_question))
        print(f"\n Model Name: {model_name} - Context Window: {num_tokens} tokens \n")
        print(user_question)
    
        # Message formatting based on model type
        if "gpt" in model_name or "o3-mini" in model_name:
            messages = [
                {"role": "system", "content": "You are an expert in NLP coreference resolution for scientific papers."},
                {"role": "user", "content": user_question}
            ]
        else: # For Ollama Implementation.
            template = ChatPromptTemplate.from_messages([
                ("system", "You are an expert in NLP coreference resolution for scientific papers."),
                ("human", "{user_input}")
            ])
            messages = template.format_messages(user_input=user_question)
    
        # Applying Coreference Resolution with progress bar
        with tqdm(total=1, desc="Applying Coreference Resolution") as pbar:
            response = llm_generator.invoke(messages)
            pbar.update(1)
    
        response_temp = response.content
    
        # Separate thinking process and final answer
        if "gpt" in model_name or "o3-mini" in model_name:
            thinking = "NA"
            response_final = response_temp  
        else:
            thinking, response_final = extract_think_text(response_temp)
    
        # Print execution time
        end_time = time.time()
        elapsed_time = round(end_time - start_time, 2)
    
        print(f"\nTotal execution time: {elapsed_time} seconds")
        
        return thinking, response_final, num_tokens,elapsed_time
    else:
        print("The mixed model parameter should be YES or NO and you have assigned: ",model_mixed)
        thinking = None
        response_final = None
        num_tokens = 0
        elapsed_time = 0.0
        
    return thinking, response_final, num_tokens, elapsed_time


        

In [11]:
# Cell 7: Coreference Resolution – Amazon Bedrock Models
#
# get_coreference_resolution_bedrock(context, model_name, model_mixed)
#
# Calls Amazon Bedrock's converse() API for the following models:
#   - anthropic.claude-3-7-sonnet  : Claude 3.7 Sonnet via Bedrock
#       * model_mixed='YES'  → Extended thinking enabled (budget: 2000 tokens,
#                              temperature must be 1; max output: 10 000 tokens)
#       * model_mixed='NO'   → Standard mode (topP=0, temperature=1)
#   - us.deepseek.r1-v1:0          : DeepSeek-R1 (temperature=0, topP=0,
#                                    max output: 32 000 tokens)
#   - mistral.mistral-large-*      : Mistral Large (uses prompt_mistral template,
#                                    temperature=0, topP=0, max output: 8 000 tokens)
#   - llama3-3                     : LLaMA 3.3 via Bedrock inference profile
#                                    (temperature=0, topP=0, max output: 128 000 tokens)
#
# The response is parsed to extract:
#   - 'reasoningContent' block → stored as 'thinking' (available for Claude extended thinking)
#   - 'text' block            → stored as 'response_final' (the resolved text)
#
# Errors (e.g. throttling, model unavailability) are caught and logged; the
# function returns None values so the main loop can continue with other papers.

def get_coreference_resolution_bedrock(context: str, model_name:str, model_mixed:str):
    
    model_mixed= model_mixed.upper()
 
    if model_mixed=="YES" or model_mixed=="NO":   
        start_time = time.time()  # Start timing
        encoding = tiktoken.get_encoding("cl100k_base")
        
        # Create a Bedrock runtime client
        bedrock = session.client(service_name='bedrock-runtime')
        
        # System prompt
        system_prompt = "You are an expert in NLP coreference resolution for scientific papers."
        system_messages = [
            {"text": system_prompt}
        ]
        
        # Prompt
        if "mistral-large" in model_name:   
            user_prompt = prompt_mistral.replace("_context_", context)
        else:
            user_prompt = prompt.replace("_context_", context)
    
        # Calculate number of tokens
        num_tokens = len(encoding.encode(user_prompt))
        print(f"\n Model Name: {model_name} - Context Window: {num_tokens} tokens \n")
    
        print(user_prompt)
    
        # Prepare the user messages for model input
        user_messages = [
            {
                "role": "user",
                "content": [{"text": user_prompt}],
            }
        ]
        
        # Initialize variables outside try block
        thinking = None
        response_final = None
        
        try:
            if "claude-3-7-sonnet" in model_name and model_mixed == "YES":
                with tqdm(total=1, desc="Applying Coreference Resolution") as pbar:
                    response = bedrock.converse(
                        modelId=model_name,
                        system=system_messages,
                        messages=user_messages,
                        inferenceConfig={
                            "maxTokens": 10000, # Max 131072
                            "temperature": 1,
                            
                            
                        },
                        additionalModelRequestFields={
                            'thinking': {
                                "type": "enabled",
                                "budget_tokens": 2000
                            }
                        }
                    )
                    pbar.update(1)
    
            elif "claude-3-7-sonnet" in model_name and model_mixed == "NO":
                with tqdm(total=1, desc="Applying Coreference Resolution") as pbar:
                    response = bedrock.converse(
                        modelId=model_name,
                        system=system_messages,
                        messages=user_messages,
                        inferenceConfig={
                            "maxTokens": 10000, # Max 131072
                            "temperature": 1,
                            "topP": 0.0
                        }
                    )
                    pbar.update(1)
    
            elif "deepseek.r1" in model_name:
                with tqdm(total=1, desc="Applying Coreference Resolution") as pbar:
                    response = bedrock.converse(
                        modelId=model_name,
                        system=system_messages,
                        messages=user_messages,
                        inferenceConfig={
                            "maxTokens": 32000, # Max 32768
                            "temperature": 0,
                            "topP": 0.0
                        },
                    )
                    pbar.update(1)
    
            elif "mistral-large" in model_name:
                with tqdm(total=1, desc="Applying Coreference Resolution") as pbar:
                    response = bedrock.converse(
                        modelId=model_name,
                        system=system_messages,
                        messages=user_messages,
                        inferenceConfig={
                            "maxTokens": 8000,
                            "temperature": 0,
                            "topP": 0.0
                        },
                    )
                    pbar.update(1)
    
            elif "llama3-3" in model_name: # It need an inference profile.
                with tqdm(total=1, desc="Applying Coreference Resolution") as pbar:
                    response = bedrock.converse(
                        modelId=model_name,
                        system=system_messages,
                        messages=user_messages,
                        inferenceConfig={
                            "maxTokens": 128000, 
                            "temperature": 0,
                            "topP": 0.0
                        },
                    )
                    pbar.update(1)
    
            # Extract content from response
            contents = response["output"]["message"]["content"]
        
            for content in contents:
                if "reasoningContent" in content:
                    thinking = content["reasoningContent"]["reasoningText"]["text"]
                if "text" in content:
                    response_final = content["text"]
        
            end_time = time.time()
            elapsed_time = round(end_time - start_time, 2)
            print(f"\nTotal execution time: {elapsed_time} seconds")
        
        except Exception as e:
            print(f"An error occurred: {str(e)}")
            thinking = None
            response_final = None
            num_tokens = 0
            elapsed_time = 0.0
        
        return thinking, response_final, num_tokens, elapsed_time
    else:
        print("The mixed model parameter should be YES or NO and you have assigned: ",model_mixed )
        thinking = None
        response_final = None
        num_tokens = 0
        elapsed_time = 0.0
        
    return thinking, response_final, num_tokens, elapsed_time


In [12]:
# Cell 8: Router Function – get_coreference_resolution()
#
# Dispatches each inference call to the correct backend:
#   - Bedrock backend  : DeepSeek-R1, Claude 3.7 Sonnet, Mistral Large, LLaMA 3.3
#   - Others backend   : GPT-4.1, o3-mini, and any Ollama-hosted model
#
# This is the single entry point used by the main experiment loop (Cell 9).
# Returns the same four values as the backend functions:
#   (thinking, response_final, model_mixed, num_tokens, elapsed_time)

def get_coreference_resolution(context: str, model_name:str,model_mixed:str):
    
    model_mixed= model_mixed.upper() 
    
    if "deepseek.r1" in model_name or "anthropic.claude-3-7-sonnet" in model_name or "mistral-large" in model_name or "llama3-3" in model_name:
        thinking, response_final, num_tokens,elapsed_time=  get_coreference_resolution_bedrock(context, model_name,model_mixed)  
    else:
        thinking, response_final, num_tokens,elapsed_time=  get_coreference_resolution_others(context, model_name,model_mixed)  
    
    return thinking, response_final, model_mixed, num_tokens,elapsed_time
   

In [13]:
# Cell 9: Main Experiment Loop
#
# Runs coreference resolution for every (model, paper) combination and
# collects results into 'all_paragraphs'.
#
# Configuration:
#   models       : List of model identifiers to evaluate (one or more).
#                  The commented-out list shows the full multi-model benchmark;
#                  the active list runs only GPT-4.1 for a quick single-model pass.
#   models_mixed : Parallel list indicating whether extended thinking is enabled
#                  ('Yes') or not ('No') for the corresponding model.
#
# Output DataFrame columns per paper per model:
#   Paper_Id        : Paper identifier
#   Thinking        : Internal reasoning text (or 'NA' for non-reasoning models)
#   Related_Work_CR : Coreference-resolved Related Work text
#   Is_Mixed_Model  : Whether extended thinking was used ('YES' / 'NO')
#   Context_Window  : Prompt size in tokens
#   Time            : Inference wall-clock time in seconds
#   Model           : Model identifier string

# models = ["gpt-4.1-2025-04-14","us.deepseek.r1-v1:0","mistral.mistral-large-2402-v1:0", "us.anthropic.claude-3-7-sonnet-20250219-v1:0","us.anthropic.claude-3-7-sonnet-20250219-v1:0"]
# models_mixed=["No","No","No","No","Yes"]

models = ["gpt-4.1-2025-04-14"]
models_mixed=["No"]



all_paragraphs = pd.DataFrame()

# Loop through the list and print each name
for model,model_mixed in zip(models,models_mixed):
    
    paragraphs_model = pd.DataFrame()
    for index, row in df.iterrows(): # Iterating the df with the Related_Work_Final
        print("### ",row['Paper_Id'], "### ", "\n")
        thinking, response, model_mixed,num_tokens, timing=get_coreference_resolution(row['Related_Work_Final'],model,model_mixed)
        paragraph = pd.DataFrame([[row['Paper_Id'],thinking, response, model_mixed,num_tokens,timing,model]], columns=["Paper_Id", "Thinking", "Related_Work_CR","Is_Mixed_Model","Context_Window","Time","Model"])
        paragraphs_model = pd.concat([paragraphs_model, paragraph], ignore_index=True)
        
    all_paragraphs = pd.concat([all_paragraphs, paragraphs_model], ignore_index=True)





        

###  paper7 ###  


 Model Name: gpt-4.1-2025-04-14 - Context Window: 2802 tokens 


<context>
Please perform coreference resolution on the scientific text provided below. Analyze the provided scientific text and replace ambiguous references with their explicit antecedents to create a version where all coreferences are fully resolved. The goal is to produce a modified text where each mention clearly and explicitly refers to its intended entity, making the text more precise and easier to understand.
</context>

<scientific text>
retractions of scientific publications are often used as indicator of misconduct and sloppy science. although the number of retractions has increased over the years, their share on the publication body is still quite small. according to cope , retractions should be made due to data fabrication, data falsification, duplication, plagiarism, honest error, and ethical problems. cope emphasizes the correction of the knowledge base and the creation of transparency as 

Applying Coreference Resolution: 100%|████████████| 1/1 [00:32<00:00, 32.35s/it]



Total execution time: 33.94 seconds
###  paper3 ###  


 Model Name: gpt-4.1-2025-04-14 - Context Window: 4376 tokens 


<context>
Please perform coreference resolution on the scientific text provided below. Analyze the provided scientific text and replace ambiguous references with their explicit antecedents to create a version where all coreferences are fully resolved. The goal is to produce a modified text where each mention clearly and explicitly refers to its intended entity, making the text more precise and easier to understand.
</context>

<scientific text>
retractions are a phenomenon of growing importance in science, but it is unclear if and to what extent they should be considered a hindrance to scientific knowledge and a symptom of falling standards, rather than a positive manifestation of scientific self-correction and integrity.[line break]. the number of retracted publications has grown from practically zero three decades ago to at least 1,970 items labeled as "retraction

Applying Coreference Resolution: 100%|████████████| 1/1 [00:46<00:00, 46.75s/it]



Total execution time: 46.78 seconds
###  paper2 ###  


 Model Name: gpt-4.1-2025-04-14 - Context Window: 1177 tokens 


<context>
Please perform coreference resolution on the scientific text provided below. Analyze the provided scientific text and replace ambiguous references with their explicit antecedents to create a version where all coreferences are fully resolved. The goal is to produce a modified text where each mention clearly and explicitly refers to its intended entity, making the text more precise and easier to understand.
</context>

<scientific text>
to identify publications that propagate unreliability, researchers have developed approaches for auditing literature that cites unreliable work, relying on the fact that not all citations are equally influential in citing publications [22]. fu and schneider [45] developed the keystone framework and proposed its use for identifying publications whose content may be significantly affected by the unreliable sources they cited. u

Applying Coreference Resolution: 100%|████████████| 1/1 [00:03<00:00,  3.96s/it]


... (5 outputs left out)

Applying Coreference Resolution: 100%|████████████| 1/1 [00:05<00:00,  5.99s/it]



Total execution time: 6.02 seconds
###  paper5 ###  


 Model Name: gpt-4.1-2025-04-14 - Context Window: 3600 tokens 


<context>
Please perform coreference resolution on the scientific text provided below. Analyze the provided scientific text and replace ambiguous references with their explicit antecedents to create a version where all coreferences are fully resolved. The goal is to produce a modified text where each mention clearly and explicitly refers to its intended entity, making the text more precise and easier to understand.
</context>

<scientific text>
retraction is intended to remove seriously flawed papers from the scientific literature, and papers are not meant to be cited after retraction [9] [9]. while intentional post-retraction citation does happen (see e.g., the example on page 13 of grieneisen and zhang 2012), citation of research discredited due to misconduct is typically thought to be accidental or careless. yet continued citation of papers retracted due to miscon

Applying Coreference Resolution: 100%|████████████| 1/1 [00:13<00:00, 13.79s/it]


Total execution time: 13.82 seconds


In [14]:
# Cell 10: Preview Results
# Display the 'all_paragraphs' DataFrame produced by the experiment loop.
# Each row is one (model, paper) inference result.
all_paragraphs


,Paper_Id,Thinking,Related_Work_CR,Is_Mixed_Model,Context_Window,Time,Model
0,paper7,NA,retractions of scientific publications are oft...,NO,2802,33.94,gpt-4.1-2025-04-14
1,paper3,NA,retractions are a phenomenon of growing import...,NO,4376,46.78,gpt-4.1-2025-04-14
2,paper2,NA,to identify publications that propagate unreli...,NO,1177,3.99,gpt-4.1-2025-04-14
3,paper6,NA,"in studies about knowledge construction, criti...",NO,4546,32.48,gpt-4.1-2025-04-14
4,paper4,NA,citation context analysis has been used to exp...,NO,1739,4.59,gpt-4.1-2025-04-14
5,paper1,NA,argumentation theory is an interdisciplinary f...,NO,1353,6.02,gpt-4.1-2025-04-14
6,paper5,NA,retraction is intended to remove seriously fla...,NO,3600,13.82,gpt-4.1-2025-04-14


In [15]:
# Cell 11: Merge Results with Paper Metadata and Sort
# Joins the coreference-resolved results ('all_paragraphs') with the original
# paper metadata ('df') on Paper_Id using an inner join.
# The result is sorted by Is_Mixed_Model, then Model, then Paper_Id to group
# runs by configuration for easier comparison.

df_final = pd.merge(df, all_paragraphs, on='Paper_Id', how='inner')  # 'inner', 'left', 'right', or 'outer'
df_final = df_final.sort_values(by=['Is_Mixed_Model','Model','Paper_Id'])
df_final

,Paper_Id,Title,Abstract,Authors,Affiliations,Year,Related_Work,Related_Work_Final,Citation_Standard,Thinking,Related_Work_CR,Is_Mixed_Model,Context_Window,Time,Model
5,paper1,towards knowledge maintenance in scientific di...,scientific digital libraries speed disseminati...,yuanxi fu; jodi schneider,"school of information sciences, university of ...",2020,argumentation theory is an interdisciplinary f...,argumentation theory is an interdisciplinary f...,[45],NA,argumentation theory is an interdisciplinary f...,NO,1353,6.02,gpt-4.1-2025-04-14
2,paper2,addressing unreliability propagation in scient...,today's scientists rely on scientific artifact...,heng zheng; yuanxi fu; m sarol; ishita sarraf;...,"grinnell college grinnell, usa; jodi schneider...",2024,to identify publications that propagate unreli...,to identify publications that propagate unreli...,[339],NA,to identify publications that propagate unreli...,NO,1177,3.99,gpt-4.1-2025-04-14
1,paper3,what difference might retractions make? an est...,the extent to which a retraction might require...,daniele fanelli; julie wong; david moher,"centre for journalology, clinical epidemiology...",2021,retractions are a phenomenon of growing import...,retractions are a phenomenon of growing import...,[332],NA,retractions are a phenomenon of growing import...,NO,4376,46.78,gpt-4.1-2025-04-14
4,paper4,continued use of retracted papers: temporal tr...,we present the first database-wide study on th...,tzu-kun hsiao; jodi schneider,"school of information sciences, university of ...",2021,citation context analysis has been used to exp...,citation context analysis has been used to exp...,[114],NA,citation context analysis has been used to exp...,NO,1739,4.59,gpt-4.1-2025-04-14
6,paper5,continued post-retraction citation of a fraudu...,this paper presents a case study of long-term ...,jodi schneider; di ye; alison hill; ashley whi...,"data science institute, columbia university, n...",2020,retraction is intended to remove seriously fla...,retraction is intended to remove seriously fla...,[8],NA,retraction is intended to remove seriously fla...,NO,3600,13.82,gpt-4.1-2025-04-14
3,paper6,critical citations in knowledge construction a...,critical citations are lacking a common defini...,frederique bordignon,"ecole des ponts, marne-la-vallée, france",2022,"in studies about knowledge construction, criti...","in studies about knowledge construction, criti...",[340],NA,"in studies about knowledge construction, criti...",NO,4546,32.48,gpt-4.1-2025-04-14
0,paper7,why do some retracted articles continue to get...,retracted publications can still receive a sub...,marion schmidt,german center for higher education research an...,2024,retractions of scientific publications are oft...,retractions of scientific publications are oft...,[338],NA,retractions of scientific publications are oft...,NO,2802,33.94,gpt-4.1-2025-04-14


In [16]:
# Cell 12: Export Final Results to Excel
# Saves the merged DataFrame to an Excel file for downstream evaluation.
# Update 'final_path' to the desired output location before running.
final_path = 'data/Coreference/Test_CoreferenceResolution_Jodiv01.xlsx'  # Update path as needed

df_final.to_excel(final_path, index=False)
